In [1]:
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
import pandas as pd

# 1. Load dataset with robust error handling
try:
    # Attempt 1: Standard load, skipping bad lines
    df = pd.read_csv("combined_news.csv", on_bad_lines='skip')
except:
    # Attempt 2: Fallback to python engine if C engine fails
    print("Standard load failed. Trying Python engine...")
    df = pd.read_csv("combined_news.csv", sep=None, engine='python', on_bad_lines='skip')

# Verify it loaded
print(f"Successfully loaded {len(df)} rows.")
print(df.head())

# 2. Prepare Data for Contrastive Learning
# SBERT needs data in a specific 'InputExample' format
# We create a list where each item is the text and its label (0 or 1)
train_examples = []
for i, row in df.iterrows():
    train_examples.append(InputExample(texts=[row['content']], label=int(row['label'])))

# 3. Create a DataLoader
# This batches the data so the model can compare multiple articles at once
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

# 4. Load the Model (The "Brain" we are going to train)
model = SentenceTransformer("all-MiniLM-L6-v2")

# 5. Define the Contrastive Loss
# BatchHardTripletLoss forces the model to separate the classes in the vector space
# It requires the embeddings to be useful for clustering
train_loss = losses.BatchHardTripletLoss(model=model)

# 6. Run Contrastive Fine-Tuning
# This is where the "reshaping" of the embedding space happens
print("Starting Contrastive Fine-Tuning...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,  # 1 or 2 epochs is usually enough for contrastive tuning
    show_progress_bar=True,
    output_path="./fine_tuned_sbert_contrastive"
)

print("Contrastive Learning Complete. Model saved to './fine_tuned_sbert_contrastive'")

Standard load failed. Trying Python engine...
Successfully loaded 8325 rows.
   label                                            content
0      1  suicide attack targets area southeast of baghd...
1      0  lying white house press secretary obama has sc...
2      1  boris johnson gives pm may advice on brexit wh...
3      1  india struggles to rein in border flows of cat...
4      1  in speech, trump tries to turn from divisive t...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,5.207000


Contrastive Learning Complete. Model saved to './fine_tuned_sbert_contrastive'


In [2]:
import shutil
from google.colab import files

# Zip the folder
# Format: shutil.make_archive(output_filename, 'zip', dir_to_zip)
shutil.make_archive('my_contrastive_model', 'zip', './fine_tuned_sbert_contrastive')

# Download the zip file
files.download('my_contrastive_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

class AttentionClassifier(nn.Module):
    def __init__(self, model_path, num_labels=2):
        super().__init__()
        # 1. Load the Contrastive-Tuned Backbone
        # We load the weights you just saved in Phase 1
        self.bert = AutoModel.from_pretrained(model_path)
        hidden_size = self.bert.config.hidden_size

        # 2. Attention Layer variables
        # This layer learns "what to look for"
        self.attention_dense = nn.Linear(hidden_size, hidden_size)
        self.attention_vector = nn.Linear(hidden_size, 1, bias=False)

        # 3. Final Classifier
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        # A. Get Word Embeddings
        # Output shape: (Batch_Size, Seq_Length, Hidden_Size)
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # We want the hidden states for all words, not just the [CLS] token
        H = outputs.last_hidden_state

        # B. Calculate Attention Weights
        # 1. Pass through a dense layer (Non-linear transformation)
        #    Math: u_t = tanh(W * h_t + b)
        u = torch.tanh(self.attention_dense(H))

        # 2. Compute importance score for each word
        #    Math: score = u_t * v
        scores = self.attention_vector(u) # Shape: (Batch, Seq_Len, 1)

        # 3. Mask padding (so we don't attend to empty space)
        scores = scores.squeeze(-1) # (Batch, Seq_Len)
        scores = scores.masked_fill(attention_mask == 0, -1e9)

        # 4. Turn scores into probabilities (0.0 to 1.0)
        alpha = F.softmax(scores, dim=1) # The "Attention Weights"

        # C. Create Context Vector
        # Weighted sum of all words based on importance
        # Shape: (Batch, Hidden_Size)
        context_vector = torch.sum(H * alpha.unsqueeze(-1), dim=1)

        # D. Classify
        logits = self.classifier(context_vector)

        # We return logits for training, and alpha for visualization later
        return logits, alpha

# 1. Define paths
# Use the folder you just created in Phase 1
contrastive_model_path = "./fine_tuned_sbert_contrastive"

# 2. Load Tokenizer
# We need the tokenizer that matches the model
tokenizer = AutoTokenizer.from_pretrained(contrastive_model_path)

# 3. Initialize our Custom Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionClassifier(contrastive_model_path).to(device)

print("Model initialized successfully!")
print(f"Running on: {device}")

Model initialized successfully!
Running on: cuda


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
from tqdm.notebook import tqdm

# --- 1. PREPARE THE DATA ---

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Split data (using the df you loaded earlier)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# Create Datasets
train_dataset = NewsDataset(
    texts=train_df.content.to_numpy(),
    labels=train_df.label.to_numpy(),
    tokenizer=tokenizer,
    max_len=128
)

val_dataset = NewsDataset(
    texts=val_df.content.to_numpy(),
    labels=val_df.label.to_numpy(),
    tokenizer=tokenizer,
    max_len=128
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# --- 2. SETUP TRAINING ---

# Initialize the model (ensure you ran the AttentionClassifier class code first)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AttentionClassifier("./fine_tuned_sbert_contrastive").to(device)

# Optimizer (AdamW is standard for Transformers)
# We use a lower learning rate because the backbone is already pre-trained
optimizer = AdamW(model.parameters(), lr=2e-5)

# Loss Function
criterion = torch.nn.CrossEntropyLoss()

# --- 3. THE TRAINING LOOP ---

EPOCHS = 3

print(f"Starting training on {device}...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    # Progress bar
    loop = tqdm(train_loader, leave=True)

    for batch in loop:
        # Move batch to GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward Pass (Get logits and attention weights)
        # Note: We ignore the attention weights [1] for now, we just need logits [0]
        logits, _ = model(input_ids, attention_mask)

        # Calculate Loss
        loss = criterion(logits, labels)

        # Backward Pass
        loss.backward()

        # Update Weights
        optimizer.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    # --- EVALUATION (After each epoch) ---
    model.eval()
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits, _ = model(input_ids, attention_mask)

            # Get predictions (Index of the highest logit)
            preds = torch.argmax(logits, dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch+1} Validation Accuracy: {val_acc:.4f}")

print("Training Complete!")

Starting training on cuda...


  0%|          | 0/417 [00:00<?, ?it/s]

Epoch 1 Validation Accuracy: 0.9988


  0%|          | 0/417 [00:00<?, ?it/s]

Epoch 2 Validation Accuracy: 0.9988


  0%|          | 0/417 [00:00<?, ?it/s]

Epoch 3 Validation Accuracy: 0.9988
Training Complete!


In [5]:
import os
import shutil
from google.colab import files

# 1. Create a directory for the full artifact
output_dir = "./final_news_classifier"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. Save the Model Weights (The "Brain")
model_save_path = os.path.join(output_dir, "attention_model_state.pt")
torch.save(model.state_dict(), model_save_path)
print(f"Model weights saved to {model_save_path}")

# 3. Save the Tokenizer (Crucial for the API to understand text)
tokenizer.save_pretrained(output_dir)
print("Tokenizer saved.")

# 4. Zip the folder for easy download
shutil.make_archive('final_news_classifier_pack', 'zip', output_dir)

# 5. Trigger Download
files.download('final_news_classifier_pack.zip')

Model weights saved to ./final_news_classifier/attention_model_state.pt
Tokenizer saved.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>